# Pruning - remove unused signals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/03_pruning/pruning.ipynb)

**Goal.** Take a trained network and **throw away the least useful weights**. A trained net
is over-parameterized: many weights barely matter. Pruning removes them, leaving a smaller
model that does almost the same job.

The parts to focus on are:

1. **The criterion** - how we score which weights are "useful" (here: magnitude).
2. **The granularity** - *unstructured* (drop individual weights) vs. *structured* (drop whole
   filters/channels).
3. **The ratio** - what fraction we remove.

The granularity choice decides whether pruning is *real*:

| approach | what is removed | size | latency |
|----------|-----------------|------|---------|
| **unstructured** | scattered single weights -> zeros | same (still a dense tensor) | same |
| **structured** | whole conv filters -> a smaller network | **smaller** | **faster** |

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.
Full run is a few minutes on a Colab GPU.

## 1. Setup

On Colab `torch`/`torchvision` are pre-installed, so nothing to `pip install`.

In [ ]:
import io, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- knobs you can play with ---
EPOCHS          = 12      # train the dense baseline
FINETUNE_EPOCHS = 4       # recover accuracy after structured pruning
BATCH_SIZE      = 128
LR              = 0.05
PRUNE_RATIO     = 0.5     # fraction of weights / filters to remove

## 2. Data - MNIST

10 classes (handwritten digits 0-9) of 28x28 grayscale images. Small and quick to download.

In [ ]:
mean = (0.1307,)
std  = (0.3081,)

train_tf = T.Compose([
    T.RandomCrop(28, padding=2),
    T.ToTensor(),
    T.Normalize(mean, std),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256,        shuffle=False, num_workers=2)

classes = train_set.classes
print(len(train_set), "train /", len(test_set), "test images")

## 3. The model

A VGG-style CNN whose channel widths are a parameter. That matters for pruning: *structured*
pruning will rebuild the **same network with narrower layers**, and `widths` is exactly what
shrinks.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )

class Net(nn.Module):               # default widths -> ~1.1M params
    def __init__(self, widths=(64, 64, 128, 128, 256, 256), num_classes=10):
        super().__init__()
        w = widths
        self.features = nn.Sequential(
            conv_block(1,    w[0]), conv_block(w[0], w[1]), nn.MaxPool2d(2),   # 28 -> 14
            conv_block(w[1], w[2]), conv_block(w[2], w[3]), nn.MaxPool2d(2),   # 14 -> 7
            conv_block(w[3], w[4]), conv_block(w[4], w[5]), nn.MaxPool2d(2),   # 7  -> 3
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(w[5], num_classes))
        self.widths = tuple(widths)

    def forward(self, x):
        return self.head(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

# the six conv blocks live at these indices inside `features` (the rest are MaxPool)
CONV_IDX = [0, 1, 3, 4, 6, 7]
def conv_blocks(model):
    return [model.features[i] for i in CONV_IDX]

print(f"params: {count_params(Net()):,}")

## 4. Train / eval / metric utilities

The same loops as the other exercises: a trainer (also reused for fine-tuning), an evaluator,
a single-image **CPU** latency probe, and an on-disk size probe.

In [ ]:
def train(model, epochs=EPOCHS, lr=LR, tag=""):
    model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        print(f"[{tag}] epoch {ep+1:02d}/{epochs}  test_acc={evaluate(model):.2f}%")
    return model

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.numel()
    model.train()
    return 100.0 * correct / total

@torch.no_grad()
def latency_ms(model, n=100):
    """Average single-image inference latency on CPU (ms)."""
    model = copy.deepcopy(model).to("cpu").eval()
    x = torch.randn(1, 1, 28, 28)
    for _ in range(10):            # warmup
        model(x)
    t0 = time.time()
    for _ in range(n):
        model(x)
    return (time.time() - t0) / n * 1000.0

def model_size_mb(model):
    """On-disk size of the saved weights (MB)."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

## 5. Train the dense baseline

Train the full-width network normally. This is the **"before"** model we prune.

In [ ]:
model_dense = train(Net(), tag="dense")
dense_acc = evaluate(model_dense)
print(f"\nDense baseline accuracy: {dense_acc:.2f}%")

## 6. The pruning recipe: criterion, granularity, ratio 🔑

Pruning is three choices:

- **Criterion** - a score for how much each weight matters. The simplest, and a strong
  baseline, is **magnitude**: a weight near zero contributes little, so $|w|$ is its
  importance. For a whole conv **filter**, the score is its **L1 norm**
  $\sum |w|$ - small-norm filters produce near-zero feature maps.
- **Granularity** - *what* we remove:
  - **unstructured**: individual weights. We set the smallest-$|w|$ ones to **zero**. The
    tensor keeps its shape - it is now a *sparse* tensor stored densely.
  - **structured**: whole filters/channels. We physically **delete** them and rebuild a
    narrower network.
- **Ratio** - the fraction removed.

Below is the criterion at the **unstructured** granularity: rank every weight by $|w|$,
zero the smallest `ratio` of them (a single **global** threshold across all layers).

In [ ]:
def prune_unstructured(model, ratio):
    """Zero the globally smallest-magnitude weights. Returns a masked copy + the threshold."""
    mp = copy.deepcopy(model)
    all_w = torch.cat([blk[0].weight.data.abs().flatten() for blk in conv_blocks(mp)])
    # TODO: threshold = the |w| at the `ratio` quantile of all weights -> torch.quantile(all_w, ratio)
    thresh = torch.zeros(())
    for blk in conv_blocks(mp):
        w = blk[0].weight.data
        # TODO: zero the small weights in place -> w.mul_((w.abs() > thresh).float())
        w.mul_((w.abs() >= 0).float())
    return mp, thresh.item()

masked, thresh = prune_unstructured(model_dense, PRUNE_RATIO)
nz  = sum((blk[0].weight.data != 0).sum().item() for blk in conv_blocks(masked))
tot = sum(blk[0].weight.numel() for blk in conv_blocks(masked))
print(f"criterion=|w|  ratio={PRUNE_RATIO}  threshold={thresh:.4f}")
print(f"actual sparsity: {1 - nz/tot:.1%}  ({tot - nz:,} of {tot:,} weights set to 0)")

# what the criterion sees: the weight-magnitude distribution and the cut
import matplotlib.pyplot as plt
all_w = torch.cat([blk[0].weight.data.abs().flatten() for blk in conv_blocks(model_dense)]).cpu()
plt.figure(figsize=(7, 3))
plt.hist(all_w.numpy(), bins=100, color="#888")
plt.axvline(thresh, color="#d9534f", lw=2, label=f"prune |w| < {thresh:.3f}")
plt.title("weight magnitudes - everything left of the line is pruned"); plt.legend(); plt.show()

## 7. Unstructured pruning: accuracy vs. how much we remove

Sweep the ratio and watch accuracy. Magnitude pruning is forgiving up to a point, then
falls off a cliff. **No fine-tuning here** - this is the raw effect of the criterion.

In [ ]:
ratios = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]
accs   = [evaluate(prune_unstructured(model_dense, r)[0].to(device)) for r in ratios]

for r, a in zip(ratios, accs):
    print(f"sparsity {r:>4.0%}  ->  accuracy {a:5.2f}%")

plt.figure(figsize=(7, 4))
plt.plot([r*100 for r in ratios], accs, "o-", color="#d9534f")
plt.xlabel("sparsity (% weights zeroed)"); plt.ylabel("test accuracy (%)")
plt.title("Unstructured magnitude pruning (no fine-tuning)"); plt.grid(alpha=0.3); plt.show()

print("\nNote: these zeros live in a DENSE tensor -> file size and latency are unchanged.")
print("A real size/speed win needs sparse storage + sparse kernels (or fine-tuning to push")
print("the ratio higher). That is why structured pruning below is the practical lever.")

## 8. Structured filter pruning - a genuinely smaller model

Now the **structured** granularity. Score each conv filter by its **L1 norm**, keep the
top `1 - ratio` per layer, and **rebuild a narrower `Net`** with only those filters.

The one subtlety is the wiring: dropping a conv's output filters means the **next** conv has
fewer input channels, and the BatchNorm in between must drop the same channels. We copy the
kept slices across:

- conv weight: keep rows (output filters) `idx`, and columns (input channels) kept by the
  previous layer;
- BatchNorm `weight / bias / running_mean / running_var`: keep `idx`;
- final `Linear`: keep the input features `idx` of the last conv.

In [ ]:
def structured_prune(model, ratio):
    """L1-norm filter pruning -> a smaller Net with the kept filters copied in."""
    blocks = conv_blocks(model)
    keep = []
    for blk in blocks:                                   # pick filters to keep, per layer
        k = max(1, round(blk[0].weight.shape[0] * (1 - ratio)))
        # TODO: score each filter by its L1 norm sum|w| over dims (1,2,3), keep the top-k:
        #       l1 = blk[0].weight.data.abs().sum(dim=(1, 2, 3)); torch.topk(l1, k).indices.sort().values
        idx = torch.arange(k)
        keep.append(idx)

    new = Net(widths=tuple(len(k) for k in keep)).eval()
    prev = torch.tensor([0])                             # input image has 1 channel
    for ob, nb, idx in zip(blocks, conv_blocks(new), keep):
        nb[0].weight.data  = ob[0].weight.data[idx][:, prev, :, :].clone()  # out=idx, in=prev
        nb[1].weight.data  = ob[1].weight.data[idx].clone()                 # BatchNorm gamma
        nb[1].bias.data    = ob[1].bias.data[idx].clone()                   # BatchNorm beta
        nb[1].running_mean = ob[1].running_mean[idx].clone()
        nb[1].running_var  = ob[1].running_var[idx].clone()
        prev = idx
    new.head[2].weight.data = model.head[2].weight.data[:, prev].clone()    # Linear in-features
    new.head[2].bias.data   = model.head[2].bias.data.clone()
    return new

pruned = structured_prune(model_dense, PRUNE_RATIO)
print(f"widths {model_dense.widths} -> {pruned.widths}")
print(f"params {count_params(model_dense):,} -> {count_params(pruned):,} "
      f"({count_params(model_dense)/count_params(pruned):.1f}x smaller)")
print(f"accuracy right after pruning (no fine-tune): {evaluate(pruned.to(device)):.2f}%")

## 9. Fine-tune to recover

Removing filters dents accuracy. A short fine-tune (a few epochs, smaller LR) lets the
remaining filters compensate - this **prune -> fine-tune** cycle is standard practice.

In [ ]:
pruned = train(pruned, epochs=FINETUNE_EPOCHS, lr=LR * 0.2, tag="finetune")
pruned_acc = evaluate(pruned)
print(f"\nStructured-pruned accuracy after fine-tune: {pruned_acc:.2f}%")

## 10. Results - size, accuracy & latency, before vs. after

Three models at the **same pruning ratio**: the dense baseline, the *unstructured* (masked)
model, and the *structured* (rebuilt + fine-tuned) model. Only structured pruning moves the
size and latency bars.

In [ ]:
masked_acc = evaluate(masked.to(device))

rows = [
    ("dense baseline",        model_size_mb(model_dense), dense_acc,  latency_ms(model_dense)),
    ("unstructured (masked)", model_size_mb(masked),      masked_acc, latency_ms(masked)),
    ("structured (rebuilt)",  model_size_mb(pruned),      pruned_acc, latency_ms(pruned)),
]

print(f"{'model':<24}{'size':>10}{'accuracy':>11}{'CPU latency':>14}")
print("-" * 59)
for name, sz, acc, lat in rows:
    print(f"{name:<24}{sz:>8.2f}MB{acc:>10.2f}%{lat:>12.2f}ms")

print(f"\n>>> structured: {model_size_mb(model_dense)/model_size_mb(pruned):.1f}x smaller, "
      f"{latency_ms(model_dense)/latency_ms(pruned):.1f}x faster, "
      f"{pruned_acc-dense_acc:+.2f}% accuracy at {PRUNE_RATIO:.0%} pruned.")
print(">>> unstructured: same size & latency - the zeros sit in a dense tensor.")

names  = [r[0] for r in rows]
sizes  = [r[1] for r in rows]
accs   = [r[2] for r in rows]
lats   = [r[3] for r in rows]
colors = ["#888", "#d9534f", "#5cb85c"]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(names, sizes, color=colors); ax[0].set_title("Model size (MB)")
ax[1].bar(names, accs,  color=colors); ax[1].set_title("Accuracy (%)"); ax[1].set_ylim(min(accs)-3, 100)
ax[2].bar(names, lats,  color=colors); ax[2].set_title("CPU latency / image (ms)")
for a in ax: a.tick_params(axis="x", rotation=12)
plt.tight_layout(); plt.show()

## 11. Takeaways & things to try

**What you saw:** magnitude is a simple, strong **criterion**; the **granularity** decides
whether pruning is real. Unstructured pruning makes weights zero but not the *tensor* smaller -
size and speed need sparse kernels. **Structured** pruning deletes whole filters, so the model
is actually smaller and faster, and a short **fine-tune** recovers the lost accuracy.

**Experiment (edit and re-run):**
- Push `PRUNE_RATIO` up (`0.7`, `0.8`). How much does fine-tuning rescue?
- Swap the **criterion**: prune *random* filters instead of low-L1. Magnitude should win.
- Prune **per-layer** vs. the **global** threshold used here - early layers are often more
  sensitive.
- **Iterative pruning**: prune a little, fine-tune, repeat - usually beats one big cut.

**Why it matters for the edge:** structured pruning is how you trade a known slice of accuracy
for a smaller, faster model. It stacks with the others: **distill**, **prune**, then
**quantize** the survivor.